# 01 — Exploratory data analysis
Feature table built by `python -m train.build_features`. Run from the repo root (or `notebooks/`).
If `data_source` prints `synthetic-test-fixture`, every number below is from the offline test fixture, not real data.

In [ ]:
import os, sys, json
from pathlib import Path
ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT)); os.chdir(ROOT)
import pandas as pd, numpy as np
import matplotlib; matplotlib.use("Agg") if "ipykernel" not in sys.modules else None
import matplotlib.pyplot as plt
from engine.io_utils import read_table
df = read_table(ROOT / "data/features.parquet")
fmeta = json.loads((ROOT / "data/features_meta.json").read_text())
print(len(df), "CVEs | data source:", fmeta["data_source"])

In [ ]:
df["kev"] = df["kev_date_added"].notna()
by_year = df.groupby(df["published"].dt.year).agg(cves=("cve", "size"), kev=("kev", "sum"))
by_year["kev_rate_%"] = (100 * by_year["kev"] / by_year["cves"]).round(2)
by_year

Label balance: KEV is rare (roughly 1-2% of CVEs), so PR-AUC and precision@k matter more than accuracy.

In [ ]:
print(f"overall KEV rate: {100*df['kev'].mean():.2f}%  |  CVEs without a CVSS v3 vector: {int(df['cvss_missing'].sum()):,}")
rates = {c: df.loc[df[c] == 1, "kev"].mean() for c in ["av_N", "pr_N", "ui_N", "kw_rce", "kw_unauth", "kw_deser", "s_C"]}
pd.Series(rates, name="KEV rate when flag set").sort_values(ascending=False)

In [ ]:
fig, ax = plt.subplots(figsize=(6, 3.5))
df.loc[~df.kev, "cvss_base"].plot.hist(bins=40, alpha=.6, density=True, ax=ax, label="not in KEV")
df.loc[df.kev, "cvss_base"].plot.hist(bins=40, alpha=.6, density=True, ax=ax, label="in KEV")
ax.set_xlabel("CVSS base score"); ax.legend(); ax.set_title("CVSS separates KEV CVEs only weakly")
fig.tight_layout()

In [ ]:
lag = (df.loc[df.kev, "kev_date_added"] - df.loc[df.kev, "published"]).dt.days
print("days from NVD publication to KEV listing: median", int(lag.median()), "| share listed >30 days after:", round(float((lag > 30).mean()), 3))